# zembed-1 Embedding API Guide

This notebook shows how to use `zclient.models.embed()` to get embedding vectors from text. We'll go through every parameter, then try two practical examples: sentence similarity and clustering.

### Prerequisites
- Python 3.12+
- `zeroentropy` SDK (`pip install zeroentropy`)
- A ZeroEntropy API key ([get one here](https://dashboard.zeroentropy.dev))
- `numpy` for dot products and cosine similarity
- `scikit-learn` for the clustering example

### Install dependencies

In [ ]:
%pip install zeroentropy numpy scikit-learn

### Set up the client

In [ ]:
from zeroentropy import ZeroEntropy
import numpy as np
import os

zclient = ZeroEntropy(api_key=os.environ["ZEROENTROPY_API_KEY"])

EMBEDDING_MODEL = "zembed-1"

## Basic usage

Pass one or more strings and get back a list of vectors. Each item in `response.results` has an `embedding` (list of floats). Results are returned in the same order as the input.

In [ ]:
response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=[
        "ZeroEntropy builds state-of-the-art retrieval models.",
        "The weather in San Francisco is foggy today.",
    ],
    input_type="document"
)

# response.results is a list of Result, one per input string (same order as input)
for i, item in enumerate(response.results):
    vec = item.embedding
    print(f"Index {i}: dim={len(vec)}, first 5 values={vec[:5]}")

print(f"\nToken usage: {response.usage.total_tokens} tokens, {response.usage.total_bytes} bytes")

## Parameters

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | `str` | Yes | Model ID, e.g. `"zembed-1"`. |
| `input` | `str \| list[str]` | Yes | One string or a list of strings to embed. |
| `input_type` | `"query" \| "document"` | Yes | Use `"query"` for search queries, `"document"` for corpus text. |
| `dimensions` | `int` | No | Vector size. Supported values (flexible projections): `40`, `80`, `160`, `320`, `640`, `1280`, `2560` (full). Defaults to `2560`. |
| `encoding_format` | `"float" \| "base64"` | No | How the vectors are returned. `"base64"` is more compact. Defaults to `"float"`. |
| `latency` | `"fast" \| "slow"` | No | `"fast"` gives sub-second responses but tighter rate limits. `"slow"` has higher throughput. If you don't set this, the API tries fast first and falls back to slow. |

## Asymmetric retrieval

In most search setups, the query is short ("What is backpropagation?") and the documents are long. `zembed-1` handles this with separate input types:

- `"query"` is tuned for short search queries
- `"document"` is tuned for longer passages

When doing retrieval, embed the query with `input_type="query"` and the corpus with `input_type="document"`. For symmetric tasks (e.g. comparing two sentences), use the same `input_type` for all inputs.

In [ ]:
query = "What is backpropagation?"
documents = [
    "Backpropagation is an algorithm for training neural networks by computing gradients of the loss function with respect to each weight.",
    "Photosynthesis is the process by which green plants convert sunlight into chemical energy.",
    "Gradient descent iteratively updates model parameters in the direction that minimises the loss.",
]

# Embed the query
q_response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=[query],
    input_type="query",
)
q_vec = np.array(q_response.results[0].embedding)

# Embed the documents
d_response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=documents,
    input_type="document",
)
d_vecs = np.array([item.embedding for item in d_response.results])

# Rank by dot-product similarity
scores = d_vecs @ q_vec
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

print(f"Query: \"{query}\"\n")
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  {rank}. (score={score:.4f}) {documents[idx][:80]}...")

## Dimension projections

`zembed-1` supports flexible dimension projections, so you can request a smaller vector without re-embedding. Supported values: `40`, `80`, `160`, `320`, `640`, `1280`, `2560` (full, default).

In [ ]:
text = ["Dimension projections let you choose your vector size at query time."]

for dim in [40, 80, 160, 320, 640, 1280, 2560]:
    response = zclient.models.embed(
        model=EMBEDDING_MODEL,
        input=text,
        input_type="document",
        dimensions=dim,
        latency="fast",
    )
    vec = response.results[0].embedding
    print(f"dimensions={dim:>5}  ->  vector length = {len(vec)}")

## Latency modes

Same idea as [`models.rerank()`](https://docs.zeroentropy.dev/api-reference/models/rerank): you can pick between fast and slow.

| Mode | Latency | Rate limit | When to use |
|------|---------|------------|-------------|
| `"fast"` | Sub-second | Lower | Real-time queries, interactive search |
| `"slow"` | A few seconds | Much higher | Batch indexing, bulk embedding jobs |

If you don't set `latency`, the API tries fast first and falls back to slow if you're over the rate limit.

In [ ]:
# Real-time query embedding (low latency)
fast_response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=["What causes ocean tides?"],
    input_type="query",
    latency="fast",
)

# Bulk document embedding (high throughput)
slow_response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=[
        "The gravitational pull of the Moon is the primary cause of ocean tides.",
        "Solar energy drives the water cycle through evaporation and precipitation.",
        "Tidal forces also cause slight deformations in the Earth's crust.",
    ],
    input_type="document",
    latency="slow",
)

## Example: sentence similarity

Embed a few sentences and compute pairwise cosine similarity. No vector database needed, just numpy.

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "A kitten was resting on the rug.",
    "The stock market surged after the Fed announcement.",
    "Equities rallied following the central bank's decision.",
]

response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=sentences,
    input_type="document",
)

vecs = np.array([item.embedding for item in response.results])

# Cosine similarity matrix
sim_matrix = vecs @ vecs.T

# Display
labels = [f"S{i}" for i in range(len(sentences))]
header = "       " + "  ".join(f"{l:>6}" for l in labels)
print(header)
for i, row in enumerate(sim_matrix):
    vals = "  ".join(f"{v:6.3f}" for v in row)
    print(f"{labels[i]:>6}  {vals}")

print("\nSentences:")
for i, s in enumerate(sentences):
    print(f"  S{i}: {s}")

## Example: clustering

Embed some sentences and run KMeans to group them by topic. No vector database, just scikit-learn.

In [ ]:
from sklearn.cluster import KMeans

sentences = [
    "Python is a popular programming language.",
    "JavaScript runs natively in the browser.",
    "Rust provides memory safety without a garbage collector.",
    "The Eiffel Tower is located in Paris.",
    "The Colosseum is an ancient amphitheatre in Rome.",
    "Machu Picchu sits high in the Andes mountains.",
]

response = zclient.models.embed(
    model=EMBEDDING_MODEL,
    input=sentences,
    input_type="document",
    dimensions=320,
)

vecs = np.array([item.embedding for item in response.results])

kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
labels = kmeans.fit_predict(vecs)

for cluster_id in sorted(set(labels)):
    print(f"\nCluster {cluster_id}:")
    for i, label in enumerate(labels):
        if label == cluster_id:
            print(f"  - {sentences[i]}")

## Recap

Here's a quick cheat sheet:

| I want to... | Parameter | Value |
|--------------|-----------|-------|
| Embed search queries | `input_type` | `"query"` |
| Embed corpus documents | `input_type` | `"document"` |
| Use smaller vectors | `dimensions` | `40`, `80`, `160`, `320`, `640`, `1280`, or `2560` |
| Get fast responses | `latency` | `"fast"` |
| Maximize throughput | `latency` | `"slow"` |